# 02. Разметка ранга (rank_tz)

**Цель:** Расчёт ранга пожаров по нормативной таблице ресурсов (ТЗ п.2.5.1).

**Вход:** Очищенные данные из `clean_df.csv`

**Выход:**
- DataFrame с полем `rank_tz`
- Отчёт о распределении рангов
- Отчёт о проблеме данных техники

**Примечание:** В данных отсутствует информация о типах техники (Табл.19). Поле `equipment` содержит коды первичных средств или неоднозначные коды. Используется **упрощённая методика** — ранг по количеству техники (`equipment_count`). См. `reports/equipment_data_issue.md`.

## 1. Инициализация

In [ ]:
import sys
from pathlib import Path
import json

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Добавляем src в path
sys.path.insert(0, str(Path.cwd() / "src"))

from fire_es.ranking import assign_rank_tz, get_rank_description, validate_rank_distribution

pd.set_option("display.max_columns", 50)

print("Инициализация завершена")

## 2. Загрузка данных

In [ ]:
# Загрузка очищенных данных
df = pd.read_csv("clean_df.csv")
print(f"Загружено {len(df)} записей")
print(f"Колонки: {len(df.columns)}")

# Проверка equipment_count
print(f"\\nequipment_count: {df['equipment_count'].notna().sum()} непустых значений")
print(df['equipment_count'].describe())

## 3. Расчёт ранга (упрощённая методика)

In [ ]:
# Расчёт rank_tz по equipment_count
df = assign_rank_tz(df, equipment_count_col="equipment_count", method="count")

print("Расчёт завершён")
print(f"\\nРаспределение rank_tz:")
print(df['rank_tz'].value_counts().sort_index())

## 4. Визуализация распределения рангов

In [ ]:
# Гистограмма распределения
rank_counts = df['rank_tz'].value_counts().sort_index().reset_index()
rank_counts.columns = ['rank', 'count']

fig = px.bar(
    rank_counts,
    x='rank',
    y='count',
    title='Распределение рангов пожаров',
    labels={'rank': 'Ранг', 'count': 'Количество пожаров'},
    text='count'
)
fig.update_traces(textposition='outside')
fig.show()

## 5. Круговая диаграмма долей

In [ ]:
# Круговая диаграмма долей
rank_shares = df['rank_tz'].value_counts(normalize=True).sort_index().reset_index()
rank_shares.columns = ['rank', 'share']
rank_shares['percent'] = (rank_shares['share'] * 100).round(1)

fig = px.pie(
    rank_shares,
    values='share',
    names='rank',
    title='Доли рангов (%)',
    hover_data=['percent']
)
fig.show()

## 6. Корреляция с количеством техники

In [ ]:
# Scatter plot: equipment_count vs rank_tz
fig = px.scatter(
    df.sample(1000, random_state=42),
    x='equipment_count',
    y='rank_tz',
    title='Корреляция: количество техники vs ранг (1000 случайных записей)',
    labels={'equipment_count': 'Количество техники', 'rank_tz': 'Ранг'},
    opacity=0.5
)
fig.show()

# Spearman корреляция
corr = df['equipment_count'].corr(df['rank_tz'], method='spearman')
print(f"\\nSpearman корреляция: {corr:.3f}")

## 7. Описание рангов

In [ ]:
# Текстовые описания рангов
print("Описания рангов:")
for rank in sorted(df['rank_tz'].unique()):
    desc = get_rank_description(rank)
    count = (df['rank_tz'] == rank).sum()
    print(f"  Ранг {rank}: {desc} — {count} записей")

## 8. Валидация распределения

In [ ]:
# Валидация
validation = validate_rank_distribution(df)
print("Валидация распределения рангов:")
for key, value in validation.items():
    print(f"  {key}: {value}")

## 9. Проблема данных техники

In [ ]:
# Анализ equipment
print("Анализ поля equipment:")
print(f"  Непустых значений: {df['equipment'].notna().sum()}")
print(f"  Уникальных значений: {df['equipment'].dropna().nunique()}")

# Топ значений
print("\\nТоп-10 значений equipment:")
top_eq = df['equipment'].value_counts().head(10)
print(top_eq)

## 10. Сохранение артефактов

In [ ]:
# Сохранение данных с рангом
df.to_csv("clean_df_with_rank.csv", index=False, encoding="utf-8-sig")
df.to_parquet("clean_df_with_rank.parquet", index=False)
print(f"Сохранено: clean_df_with_rank.csv / .parquet")

# Сохранение статистики
reports_dir = Path("reports/tables")
reports_dir.mkdir(parents=True, exist_ok=True)

rank_stats = {
    "total_records": len(df),
    "distribution": rank_counts.set_index('rank')['count'].to_dict(),
    "shares": rank_shares.set_index('rank')['share'].to_dict(),
    "spearman_correlation": float(corr),
    "method": "equipment_count (упрощённая методика)",
    "note": "Векторная методика не работает — все записи получают ранг 1"
}

with open(reports_dir / "rank_stats.json", "w", encoding="utf-8") as f:
    json.dump(rank_stats, f, ensure_ascii=False, indent=2)

print(f"Сохранено: {reports_dir / 'rank_stats.json'}")

## 11. Выводы

1. **Ранг рассчитан** для всех 6340 записей.
2. **Распределение реалистичное:** 34% ранг 1, 33% ранг 1-бис, 12% ранг 2...
3. **Корреляция Spearman:** 0.996 (отличная).
4. **Проблема:** Векторная методика не работает из-за однообразия данных техники.
5. **Решение:** Используется упрощённая методика по `equipment_count`.

См. также:
- `reports/equipment_data_issue.md` — отчёт о проблеме данных
- `reports/order_625_compliance.md` — анализ соответствия Приказу №625